# 第5讲：次数、概率与极端风险：怎样选择模型（学生操练）

        > 课程：《交通大数据分析与应用》  
        > 数据：课程模拟数据，不是实际监测数据  
        > 建议用时：7分钟

        ## 目标

        1. 计算3÷96和2÷48，并说明为什么不能只比较总次数；
2. 把PROB_THRESHOLD改为0.35和0.65，记录漏报、误报和报警数怎样变化；
3. 面对14个日最大通行时间，写出一个可以说和一个不能说的结论；

        代码可以直接运行；请按`TODO`修改参数、核对输出并完成解释。

## 1. Setup｜环境、路径与参数

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})
RANDOM_STATE = 42
print("环境已就绪；数据目录：", DATA_DIR.resolve())

## 2. 读取15分钟记录并识别一行数据

In [ ]:
traffic = pd.read_csv(DATA_DIR / "traffic_15min.csv", parse_dates=["timestamp"])
traffic = traffic.sort_values(["detector_id", "timestamp"]).reset_index(drop=True)
example = traffic.loc[
    (traffic["detector_id"] == "D05") &
    (traffic["timestamp"].between("2026-03-02 08:15", "2026-03-02 08:45")),
    ["timestamp", "detector_id", "flow_15min", "speed_kmh", "occupancy_pct", "severe_congestion", "travel_time_min"],
]
display(example)
print("一行数据：一个检测器在一个15分钟时段内的汇总状态")
print("时间范围：", traffic["timestamp"].min(), "至", traffic["timestamp"].max())

## 3. 构造每日次数并比较暴露量

In [ ]:
manual_rate = pd.DataFrame({
    "detector": ["A", "B"],
    "count": [3, 2],
    "observed_intervals": [96, 48],
})
manual_rate["rate_per_interval"] = manual_rate["count"] / manual_rate["observed_intervals"]
display(manual_rate)
print("A发生率：", round(manual_rate.loc[0, "rate_per_interval"] * 100, 3), "%")
print("B发生率：", round(manual_rate.loc[1, "rate_per_interval"] * 100, 3), "%")

daily = (traffic.assign(date=traffic["timestamp"].dt.date)
         .groupby(["date", "detector_id"], as_index=False)
         .agg(severe_intervals=("severe_congestion", "sum"),
              exposure_intervals=("severe_congestion", "size"),
              rain=("rain", "max"),
              is_weekend=("is_weekend", "max")))
display(daily.head(8))
daily.to_csv(OUTPUT_DIR / "lesson05_daily_counts.csv", index=False)

## 4. 拟合Poisson并读取IRR与离散比

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

count_fit = smf.glm(
    "severe_intervals ~ rain + is_weekend", data=daily,
    family=sm.families.Poisson(),
    offset=np.log(daily["exposure_intervals"]),
).fit()
pearson_dispersion = float(np.sum(count_fit.resid_pearson ** 2) / count_fit.df_resid)
count_result = pd.DataFrame({
    "coefficient": count_fit.params,
    "IRR": np.exp(count_fit.params),
}).round(3)
display(count_result)
print({"detector_days": len(daily),
       "exposure_values": sorted(daily["exposure_intervals"].unique().tolist()),
       "observed_mean": round(daily["severe_intervals"].mean(), 3),
       "observed_variance": round(daily["severe_intervals"].var(), 3),
       "pearson_dispersion": round(pearson_dispersion, 3)})
print("解释重点：IRR说明相对发生率；离散比用于判断Poisson是否低估波动。")

## 5. 构造下一时段标签并按时间切分

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

features = ["flow_15min", "speed_kmh", "occupancy_pct", "rain", "is_peak"]
alert = traffic.copy()
alert["target_timestamp"] = alert.groupby("detector_id")["timestamp"].shift(-1)
alert["next_severe"] = alert.groupby("detector_id")["severe_congestion"].shift(-1)
alert = alert.dropna(subset=["target_timestamp", "next_severe"]).copy()
alert["next_severe"] = alert["next_severe"].astype(int)

unique_times = np.sort(alert["timestamp"].unique())
split_time = pd.Timestamp(unique_times[int(len(unique_times) * .75)])
train = alert.loc[alert["target_timestamp"] < split_time].copy()
test = alert.loc[alert["timestamp"] >= split_time].copy()
clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
).fit(train[features], train["next_severe"])
test["risk"] = clf.predict_proba(test[features])[:, 1]
pair_example = alert.loc[
    (alert["detector_id"] == "D05") & (alert["timestamp"] == "2026-03-02 08:15"),
    ["timestamp", "target_timestamp", "detector_id", "speed_kmh", "occupancy_pct", "next_severe"],
]
display(pair_example)
print({"split_time": str(split_time),
       "train_current_max": str(train["timestamp"].max()),
       "train_target_max": str(train["target_timestamp"].max()),
       "test_current_min": str(test["timestamp"].min()),
       "train_rows": len(train), "test_rows": len(test),
       "test_positive_rate": round(test["next_severe"].mean(), 3)})

## 6. 比较阈值并落实每时段Top-3

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

def evaluate_threshold(frame, threshold):
    alarm = (frame["risk"] >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(frame["next_severe"], alarm, labels=[0, 1]).ravel()
    alarms_per_time = pd.Series(alarm.to_numpy(), index=frame["timestamp"]).groupby(level=0).sum()
    return {"threshold": threshold, "missed": int(fn), "false_alarm": int(fp),
            "precision": round(precision_score(frame["next_severe"], alarm, zero_division=0), 3),
            "recall": round(recall_score(frame["next_severe"], alarm, zero_division=0), 3),
            "max_alarms_per_time": int(alarms_per_time.max())}

PROB_THRESHOLD = 0.50  # TODO：改为0.35和0.65，比较漏报、误报和报警数
threshold_table = pd.DataFrame(evaluate_threshold(test, value) for value in [0.35, PROB_THRESHOLD, 0.65])
threshold_table = threshold_table.drop_duplicates("threshold").sort_values("threshold")
display(threshold_table)
threshold_table.to_csv(OUTPUT_DIR / "lesson05_threshold_comparison.csv", index=False)

example_time = pd.Timestamp("2026-03-12 16:30")
example_risk = test.loc[test["timestamp"] == example_time,
                        ["timestamp", "detector_id", "speed_kmh", "occupancy_pct", "risk", "next_severe"]]
display(example_risk.sort_values("risk", ascending=False).round(3))

top3 = (test.sort_values(["timestamp", "risk"], ascending=[True, False])
        .groupby("timestamp", group_keys=False).head(3).copy())
print({"每时段最多位置数": int(top3.groupby("timestamp").size().max()),
       "阈值0.5漏报": int(threshold_table.loc[threshold_table["threshold"] == 0.5, "missed"].iloc[0]),
       "阈值0.5误报": int(threshold_table.loc[threshold_table["threshold"] == 0.5, "false_alarm"].iloc[0])})
top3[["timestamp", "target_timestamp", "detector_id", "risk", "next_severe"]].to_csv(
    OUTPUT_DIR / "lesson05_top3_alerts.csv", index=False)

## 7. 构造每日最大通行时间并说明限制

In [ ]:
daily_extreme = (traffic.assign(date=traffic["timestamp"].dt.date)
                 .groupby(["detector_id", "date"], as_index=False)
                 .agg(daily_mean_travel_time=("travel_time_min", "mean"),
                      daily_max_travel_time=("travel_time_min", "max")))
d05_extreme = daily_extreme.loc[daily_extreme["detector_id"] == "D05"].copy()
display(d05_extreme.head(7).round(2))
daily_extreme.to_csv(OUTPUT_DIR / "lesson05_daily_max_travel_time.csv", index=False)
print({"D05_days": len(d05_extreme),
       "D05_largest_observed_travel_time": round(d05_extreme["daily_max_travel_time"].max(), 2)})
print("限制：只有14个日最大值。本讲不拟合GEV，也不报告长期返回水平。")

checks = {
    "offset已使用": count_fit.model.offset is not None,
    "训练目标早于检验输入": train["target_timestamp"].max() < test["timestamp"].min(),
    "风险在0至1": test["risk"].between(0, 1).all(),
    "每时段不超过3个": top3.groupby("timestamp").size().le(3).all(),
    "D05日最大值共14个": len(d05_extreme) == 14,
    "三个输出文件已生成": all((OUTPUT_DIR / name).exists() for name in [
        "lesson05_daily_counts.csv", "lesson05_threshold_comparison.csv",
        "lesson05_top3_alerts.csv", "lesson05_daily_max_travel_time.csv"]),
}
status = "PASS" if all(checks.values()) else "CHECK"
(OUTPUT_DIR / "自检结果.txt").write_text(status, encoding="utf-8")
print(status, checks)

## Checks｜当堂记录

        - 计算3÷96和2÷48，并说明为什么不能只比较总次数
- 把PROB_THRESHOLD改为0.35和0.65，记录漏报、误报和报警数怎样变化
- 面对14个日最大通行时间，写出一个可以说和一个不能说的结论

        **预期结果：** 发生率比较、阈值比较表、Top-3清单和每日最大通行时间表。

        **完成标准：** 能说清一行数据、目标变量和时间方向；结果用交通管理语言解释；Notebook生成PASS。

        请在课堂记录中写下：改了什么参数、结果发生了什么变化、这个变化在交通问题中意味着什么。